# OpenAI Responses API와 프롬프트 작성

OpenAI Python SDK로 요청 한 건을 보내고 응답에서 글을 꺼냅니다. 같은 입력을 두고 지시문만 바꾸어 답이 어떻게 달라지는지도 확인합니다.

| 함께 쓰는 수업 시간 | 결과를 확인할 장면 |
|---|---|
| 첫 호출과 응답 읽기 6시간 | 사과를 영어로 옮기는 요청 한 건을 보냅니다. 이어서 공지 C1·C2로 장소 변경을 확인합니다. 응답 객체를 읽는 작업과 새로 요청하는 작업을 구별합니다. |
| 프롬프트 작성 실험 10시간 | 8장 끝의 M1 개인 편집부터 9장까지 이어집니다. 독자·목적을 정하고 시점이 다른 원문을 함께 주며, 마감·예외·길이가 충돌하는 실제 답을 고칩니다. |
| 예시·형식·정보 부족 처리 8시간 | 모임 접수표를 설계하고 예시가 새 자료에도 통하는지 확인한 뒤, 시설 점검 자료를 짝에게 넘깁니다. |

**준비물**은 노트북과 [개인 편집 활동지](practice.md)입니다. 실습에 쓸 메모와 공지는 본문에 있습니다. C1~C5 본문에는 2026년 11월 26일 실제로 받은 대표 응답이 보관되어 있습니다. 실행 응답이 비어 있는 새 실험에서는 자신의 답을 받아 채우세요. 접힌 모범 답안에는 실제 관찰과 사람이 원문으로 작성한 예시를 구분해 두었습니다.

### 내 결과를 다음 노트북에서 여는 준비 셀

수업 묶음의 압축을 풀고 노트북을 여세요. 이 셀은 묶음 맨 위에 `my-course` 폴더를 만듭니다. 같은 이름의 파일에 다시 저장하면 그 단계의 결과가 갱신됩니다. 뒤의 장에서 시작한다면 이 준비 셀과 해당 장의 예제를 실행하세요. 앞 장 파일이 없을 때 `read_work`의 두 번째 인자가 출발 예시가 됩니다. 낱개로 받은 노트북은 모두 같은 폴더에 둡니다.

In [ ]:
from pathlib import Path
import json

here = Path.cwd()
# 과정 폴더에서 열거나, 묶음 맨 위의 일자별 사본을 열어도 같은 장소를 씁니다.
course_root = here.parent.parent if here.parent.name == "pre-mid" else here
work = course_root / "my-course"
work.mkdir(exist_ok=True)

def save_work(name, value):
    (work / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def read_work(name, example):
    path = work / name
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return example

## 8장 · LLM과 트랜스포머를 이해하고 첫 요청 보내기

`client.responses.create()`는 모델에 요청을 보내고 응답 객체를 돌려줍니다. `input`에 보낼 글을 넣고 `reply.output_text`로 답을 읽습니다.

### 노트북에서 호출 준비하기

수업용 Python 3.10 이상 커널과 OpenAI API를 사용할 수 있는 키가 필요합니다. API는 프로그램이 정해진 방식으로 요청을 보내고 결과를 받는 접점입니다. 이 수업에서는 Python으로 글과 지시를 보내고 생성된 답을 받습니다. SDK는 이 과정을 도와주는 패키지이고, client는 요청을 보낼 때 사용하는 객체입니다. API 키는 요청을 어느 계정이 보냈는지 인증합니다. 아래 입력 칸에 입력한 키는 현재 커널의 환경에만 남습니다. 처음 사용하는 커널에서는 새 코드 셀에 아래 설치 명령을 한 번 실행하고 커널을 다시 시작하세요. 이미 설치했다면 바로 준비 셀로 갑니다.

```python
%pip install openai==3.35.1
```

수업 환경에 OPENAI_API_KEY가 설정되어 있으면 자동으로 읽습니다. 개인 노트북에서는 아래 셀의 숨김 입력란에 키를 붙여 넣고 Enter를 누르세요. 빈 셀에 키를 그대로 적어 저장하면 파일을 받은 사람에게도 보입니다. 준비 셀만으로 모델을 호출하지는 않습니다.

`OpenAI()`는 이후 요청에 사용할 클라이언트를 만듭니다. 모델에 질문을 보내는 줄은 다음 셀에 있습니다. `os.getenv("OPENAI_API_KEY")`는 설정된 키를 찾고, 키가 비어 있을 때만 `getpass`가 화면에 글자를 표시하지 않고 입력을 받습니다. 다음 셀에서 실제 질문을 보낼 때 같은 클라이언트를 사용합니다.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(course_root / ".env")
load_dotenv(here / ".env")

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키 입력: ")  # 입력한 글자는 표시되지 않습니다.
client = OpenAI()
model = "gpt-4.1-mini"  # 수업에서 사용할 수 있는 모델 이름을 지정합니다.

In [ ]:
reply = client.responses.create(model=model, input="사과를 영어 단어 하나로 번역해 줘.", store=False)
print(reply.output_text)

예를 들어 셀 아래에 apple이 보입니다. reply에는 요청의 응답 전체가 들어 있고, reply.output_text가 그 안에서 답으로 생성된 글을 꺼냅니다. 입력의 ‘사과’를 ‘포도’로 바꿔 같은 셀을 실행하면 새 요청을 보냅니다.

### LLM은 어떻게 문장을 쓸까요?

“비가 와서 우산을 ___.” 빈칸에 ‘폈다’도, ‘챙겼다’도 자연스럽습니다. **대규모 언어 모델(LLM, Large Language Model)**은 많은 글로 학습하면서 이런 문맥에서 어떤 표현이 이어질 가능성이 큰지 배웁니다. 이 수업에서 다루는 생성형 언어 모델은 많은 글로 미리 배우는 단계(사전 학습)에서 글의 앞부분으로 다음 조각을 예측하고, 원문에 실제로 나온 조각의 확률이 높아지도록 모델 안의 수많은 수치(매개변수)를 조정합니다.

모델이 다루는 글의 조각을 **토큰**이라고 합니다. 한 단어가 한 토큰이 될 수도 있고 여러 토큰으로 나뉠 수도 있습니다. 모델은 답을 쓸 때 다음 토큰 후보마다 확률을 계산해 하나를 고릅니다. 선택한 토큰까지 문맥에 넣고 다음 토큰을 고르는 일을 반복합니다. 같은 질문에도 ‘우산을 폈다’와 ‘우산을 펼쳤다’처럼 표현이 달라질 수 있는 이유입니다.

이 수업에서 공지를 입력하면 모델은 학습해 둔 매개변수를 이용해 **지금 보낸 공지와 요청에 이어질 답**을 만듭니다. 공지의 장소를 201호에서 305호로 고쳐 보내는 것은 이번 요청에 쓸 근거를 바꾸는 일입니다.

### 트랜스포머는 앞 문장을 어떻게 활용할까요?

트랜스포머(Transformer)는 오늘날 많은 LLM이 쓰는 신경망 구조입니다. 토큰을 숫자들의 묶음으로 바꾸고 순서 정보도 반영합니다. 그런 다음 여러 층에서 다른 위치의 정보를 섞어 각 위치가 문맥에서 어떤 뜻을 갖는지 표현합니다. 이때 **어떤 위치의 정보를 얼마나 반영할지** 계산하는 방법이 셀프 어텐션(self-attention)입니다.

“민지가 배를 깎았다. 여기서 배는 무엇일까?”를 읽으면 ‘깎았다’가 과일을 떠올리게 합니다. “민지가 배를 타고 섬에 갔다”로 바꾸면 ‘타고’와 ‘섬’이 탈것이라는 단서가 됩니다. 답을 생성하는 위치에서는 질문 전체가 앞에 놓여 있으므로, 같은 ‘배’가 들어 있어도 함께 들어온 단서에 따라 다음 답의 후보가 달라질 수 있습니다. 이 예는 사람이 읽을 수 있는 단서를 보여 줍니다. 모델 내부에서 측정한 어텐션 점수는 아닙니다.

공지에서도 마찬가지입니다. “신입 회원은 이어폰을 챙기고, 진행자는 20분 먼저 오세요”라는 같은 원문에 **누구에게 보낼 글인지** 덧붙여 보세요. ‘신입 회원에게’라는 요청에는 이어폰이, ‘진행자에게’라는 요청에는 도착 시간이 유용한 단서입니다. 다음 실습에서 실제 응답이 그 차이를 살렸는지 확인합니다.

### “그 시간”은 어느 시각일까요?

“신청은 오후 6시에 끝납니다. 모임은 오전 10시에 시작합니다. **그 시간에** 집결지로 오세요.” 마지막 문장의 ‘그 시간’만 떼어 읽으면 시각을 정할 수 없습니다. 앞뒤를 함께 읽으면 ‘집결’과 ‘모임 시작’의 관계가 단서가 됩니다. 트랜스포머의 **어텐션**은 이런 식으로 각 위치가 다른 위치의 정보를 얼마나 반영할지 계산하는 장치입니다. 실제 모델이 이 문장에서 쓴 가중치를 우리가 측정한 것은 아닙니다. 이 예시는 문맥이 왜 필요한지 읽어 보기 위한 것입니다.

같은 마지막 문장 앞에 “신청자 확인은 오후 5시에 시작합니다”를 더해 보세요. 사람도 ‘그 시간’의 대상을 다시 물을 수 있습니다. 모델이 자연스러운 한 시각을 골랐다는 이유만으로 업무상 의도가 확인되지는 않습니다. 내가 **추측을 허용할 부분**과 **원문 근거가 필요한 부분**을 구별해 요청해야 합니다.

**직접 해 보기**. “지난번과 같은 장소로 오세요”만 받은 사람에게 구체적인 방 이름을 쓸 수 있을까요? 현장 사진이나 이전 공지처럼 내가 보았지만 이번 요청에는 넣지 않은 자료를 종이에 적습니다. 다음 장에서는 이 자료를 실제로 추가해 답이 어떻게 바뀌는지 보겠습니다.

### 공지에 없는 답도 자연스럽게 들릴까요?

공지에는 “금요일 오후 4시에 첫 모임을 합니다”라고만 적혀 있습니다. 모델이 “금요일 오후 4시, 가온관 201호에서 만나요”라고 썼다면 어느 부분을 그대로 보내도 될까요? 직접 원문에서 확인할 수 있는 부분에만 밑줄을 그어 보세요.

<details><summary>판단 예시 보기</summary><p>요일과 시간은 공지에서 확인할 수 있습니다. 장소는 알 수 없습니다. ‘가온관 201호’가 문장에 자연스럽게 이어져도 이 모임의 장소라는 근거는 없습니다.</p><p>그래서 첫 요청을 쓸 때는 원문을 함께 주고 “공지에 장소가 없으면 ‘미기재’라고 써 줘”처럼 빈 정보의 처리 방법을 정합니다. 답을 받은 뒤에는 원문에 있는 장소도 제대로 옮겼는지 확인합니다.</p></details>

### responses.create의 instructions와 input 나누기

프롬프트는 모델에게 건네는 요청입니다. 이 예제에서는 instructions에 할 일을 적고, input에 실제로 읽을 공지를 넣습니다. 말투나 독자만 바꾸고 싶을 때는 공지를 다시 쓸 필요가 없습니다. 다음 셀을 한 번 실행하고 답이 돌아올 때까지 기다리세요. 모델을 부르는 셀을 다시 실행하면 새 요청과 사용량이 발생합니다.

`client.responses.create(...)`가 API로 요청을 보내는 줄입니다. `model`에는 사용할 모델 이름, `instructions`에는 답을 작성하는 기준, `input`에는 이번에 읽을 원문을 넣습니다. `store=False`는 나중에 API로 이 응답을 조회하기 위한 저장을 끕니다. 돌아온 `response`에는 답과 사용량이 함께 있고, `response.output_text`로 답의 글만 꺼냅니다.

In [ ]:
notice = '5월 21일 금요일 오후 4시에 가온관 201호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.'

instruction = '처음 오는 회원이 모임을 찾아올 수 있도록 날짜, 시간, 장소를 한 문장으로 안내해 주세요.'
response = client.responses.create(
    model=model,
    instructions=instruction,  # 모델이 해야 할 일
    input=notice,               # 이번에 읽을 공지 원문
    store=False,
)
print(response.output_text)  # 답변 본문만 읽습니다.

소리채집 동아리 첫 모임은 5월 21일 금요일 오후 4시에 가온관 201호에서 열리니, 처음 오시는 회원은 꼭 참석해 주세요.


실제 답의 ‘5월 21일 금요일 오후 4시’와 ‘가온관 201호’를 원문에서 찾아보세요. 날짜와 장소는 맞습니다. 끝의 ‘꼭 참석해 주세요’는 모델이 덧붙인 권유입니다. 참석을 독려할 권한이 없는 안내라면 지시문에 “참석 권유 문구는 생략하세요”를 더해 시험해 보세요. <mark>답을 읽을 때는 빠진 사실과 새로 생긴 주장을 따로 찾으세요.</mark>

In [ ]:
print(response.model)  # 방금 답한 모델
print(response.usage.input_tokens, response.usage.output_tokens)  # 보낸 양과 받은 양

gpt-4.1-mini-2025-04-14
108 45


108과 45는 각각 입력과 출력의 토큰 수입니다. 토큰은 모델이 글을 처리할 때 사용하는 단위입니다. 이 셀은 앞에서 받은 응답을 읽으므로 새 모델 요청을 보내지 않습니다. 처음 요청에서 AuthenticationError가 나오면 키 입력을 확인하세요. 모델 접근 오류라면 수업 계정이 사용할 수 있는 모델명을 강사와 확인합니다.

## 방금 받은 답 한 건의 사용량 읽기

요청이 끝나면 응답에는 글과 함께 사용량 기록이 돌아옵니다. 입력에는 공지 원문 외에 모델에 보낸 지시와 메시지 형식도 포함됩니다. 출력은 모델이 생성한 토큰입니다. 한글 한 글자, 한 어절, 한 토큰은 각각 다른 단위이므로 “50자 이하”를 요청했다고 해서 출력 사용량이 50이 되는 것은 아닙니다.

In [ ]:
# 앞 셀에서 받은 응답을 다시 읽습니다. 이 셀은 모델을 호출하지 않습니다.
print("입력", response.usage.input_tokens)
print("출력", response.usage.output_tokens)
print("합계", response.usage.total_tokens)

입력 108
출력 45
합계 153


108은 모델이 읽은 쪽, 45는 쓴 쪽입니다. 합계 153은 이번 한 건의 사용량입니다. 이제 공지 전문을 다시 보낸 뒤 “장소만 한 단어로”라고 요청한다고 생각해 보세요. 입력과 출력 중 어느 쪽이 줄 것으로 예상하나요? 짝에게 이유를 설명한 뒤 앞의 호출 셀에서 지시만 바꾸어 자기 응답의 두 숫자를 확인하세요.

<details class="notebook-answer"><summary>모범 판단 펼치기</summary><p>출력이 짧아질 가능성이 큽니다. 여전히 공지 전문을 보내므로 입력에는 원문 전체가 포함됩니다. 짧게 답하라는 지시와 답변 길이의 상한을 정하는 설정은 별개입니다. 14장에서는 둘을 따로 바꿔 봅니다.</p></details>

### 호출을 고칠지, 지시를 고칠지 구분하기

코드가 실행되는 동안 Python은 어떤 모델에 어떤 일을 맡길지 담아 서버에 요청합니다. 응답을 받으면 `output_text`에서 글을 읽고, `usage`에서 이 요청이 사용한 토큰 수를 확인합니다. 같은 응답을 두 번 출력하는 일과 같은 원문을 두 번 보내는 일은 다릅니다. 셀을 다시 누르기 전, 지금 다시 실행하려는 줄에 `responses.create`가 있는지 찾아보세요.

| 눈앞에서 본 일 | 먼저 확인할 것 | 고친 뒤 확인할 결과 |
|---|---|---|
| 패키지를 가져오는 줄에서 멈춤 | 노트북을 실행하는 커널에 패키지를 설치했는가 | 첫 import가 끝남 |
| 인증 오류가 남 | 키가 있는 환경에서 만든 클라이언트인가 | 키를 노출하지 않고 한 건의 응답을 읽음 |
| 답을 받았으나 이전 장소가 나옴 | 방금 실행한 셀의 input에 어느 공지 변수가 들어갔는가 | 305호가 있는 원문과 답을 한 쌍으로 대조함 |
| 형식은 완벽하지만 원문에 없는 준비물을 넣음 | 원문 어느 문장이 그 준비물을 뒷받침하는가 | 지시를 바꿔 같은 원문으로 시험함 |

**짝과 진단하기**. 한 명은 C1의 `response.output_text`만 다시 출력하고 다른 한 명은 C2를 넣어 새로 호출합니다. 두 화면에서 같은 날짜가 보여도 어느 쪽이 새 장소를 읽을 기회가 있었는지 설명해 보세요. 코드에 키나 전체 환경 변수를 출력할 필요는 없습니다.

### 실습 · 장소가 바뀐 공지에 같은 요청 보내기

아래 공지에서 201호가 305호로 바뀌었습니다. 앞 셀의 지시는 유지하고 두 번째 응답을 받는 코드를 직접 작성하세요. 활동지에 답에서 읽은 날짜·시간·호수를 적고 첫 답과 비교합니다.

In [ ]:
new_notice = '5월 21일 금요일 오후 4시에 가온관 305호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.'
# 같은 instruction과 새 공지 new_notice를 넣어 모델을 호출하세요.

<details><summary>모범 프롬프트와 응답 살펴보기</summary><pre><code>updated = client.responses.create(
    model=model, instructions=instruction, input=new_notice, store=False
)
print(updated.output_text)</code></pre><p>이 지시로 받은 실제 응답</p><pre>소리채집 동아리 첫 모임은 5월 21일 금요일 오후 4시에 가온관 305호에서 열리니, 처음 오시는 회원께서는 이어폰을 꼭 챙겨 와 주세요.</pre><p>두 번째 응답은 305호를 안내합니다. 원문에서 바뀌지 않은 날짜와 시간도 유지됐습니다. 응답마다 곁들인 안내는 달라질 수 있습니다.</p></details>

### 신청 안내에 필요한 사실 고르기

**운영진 회의 메모(M1)**

> 6월 14일 오전 10시에 녹음 산책을 시작한다. 참가 신청은 6월 12일 오후 6시까지 동아리 게시판에서 받는다. 지난 활동에서 이미 신청서를 낸 회원은 다시 신청할 필요가 없다. 집결지는 지난번과 같다. 진행자는 시작 30분 전에 도착해 이름표를 꺼내 둔다. 비가 오면 출발 여부를 당일 오전 8시 게시판에 공지한다.

처음 신청하는 회원에게 보낼 문장을 써 보세요. 메모에서 신청할 곳과 마감을 찾으세요. 이미 신청서를 낸 회원도 같은 안내를 따라야 할까요?

### 실습 · 원문의 마감 시각만 바꿔 비교하기

위 회의 메모를 `input`에 넣고 “처음 신청하는 회원에게 신청 방법과 마감을 한 문장으로 알려 줘”라고 요청하세요. 두 번째 요청에서는 메모의 ‘오후 6시’만 ‘오후 5시’로 바꾸세요. 두 답에서 신청 마감이 바뀌었는지, 산책 시작 시각을 신청 마감으로 잘못 쓴 곳은 없는지 확인하세요.

<details><summary>판정 예시 열기</summary><p>M1의 시작은 6월 14일 오전 10시, 신청 마감은 6월 12일 오후 6시입니다. 마감만 오후 5시로 바꿨다면 새로운 안내에서 오후 5시가 신청에 붙어 있어야 합니다. 행사 자체가 6월 12일에 열린다고 변하면 원문을 다시 대조합니다.</p><p>처음 오는 회원에게 신청을 안내한다면 “기존 신청자를 제외한 회원은 6월 12일 오후 6시까지 동아리 게시판에서 녹음 산책을 신청하세요”도 가능합니다. 사람이 원문으로 작성한 예시입니다. 날짜가 같더라도 날짜에 연결된 행동이 다르면 정답이 될 수 없습니다.</p></details>

### 장소를 고친 원문과 요청 기준 보관하기

8장에서는 공지 한 건을 고쳐 다시 보냈습니다. 내가 고친 원문을 다음 시간에 그대로 써야 지시만 바꿨을 때 답의 차이를 볼 수 있습니다.

In [ ]:
save_work("notice-08.json", {"source": new_notice, "instruction": instruction})
print(new_notice == read_work("notice-08.json", {})["source"])

True


True면 저장한 원문이 글자 단위로 같습니다. 다음 수업에서 출력이 달라져도 어느 장소를 줬는지 원문으로 확인할 수 있습니다.

## 9장 · 독자와 목적을 정해 프롬프트 쓰기

9장부터 실행한다면 노트북 맨 위의 파일 준비 셀과 8장의 OpenAI 클라이언트 준비 셀을 먼저 실행하세요.

### 같은 메모로 독자별 안내 쓰기

`input`에는 메모를 넣고, `instructions`에는 **누가 읽고 무엇을 판단하거나 준비해야 하는지** 적으세요.

**운영진 회의 메모(M1)**

> 6월 14일 오전 10시에 녹음 산책을 시작한다. 참가 신청은 6월 12일 오후 6시까지 동아리 게시판에서 받는다. 지난 활동에서 이미 신청서를 낸 회원은 다시 신청할 필요가 없다. 집결지는 지난번과 같다. 진행자는 시작 30분 전에 도착해 이름표를 꺼내 둔다. 비가 오면 출발 여부를 당일 오전 8시 게시판에 공지한다.

**이미 신청한 회원**  
지시문 예시　“지난 활동에서 신청서를 낸 회원에게 다시 신청해야 하는지 한 문장으로 알려 줘.”  
답 예시　“지난 활동에서 이미 신청서를 냈다면 다시 신청하지 않아도 됩니다.”

**현장 진행자**  
지시문 예시　“진행자에게 몇 시까지 도착해서 무엇을 준비해야 하는지 한 문장으로 알려 줘.”  
답 예시　“오전 9시 30분까지 도착해 이름표를 꺼내 두세요.”

진행자의 도착 시각은 메모에 적힌 시작 시각인 오전 10시에서 30분을 빼면 구할 수 있습니다.

### ask에서 입력을 고정하고 지시문만 바꾸기

이제 8장에서 저장한 소리채집 동아리 첫 모임 공지를 세 번 보냅니다. 처음에는 읽을 사람을 지정하지 않고, 다음에는 신입 회원, 마지막에는 진행자를 지정합니다. 호출 코드에서 같은 부분은 ask 함수에 모았습니다. 첫 인자는 instructions로, 두 번째 인자는 input으로 보냅니다. 함수가 돌려주는 값은 화면에 표시할 답변 텍스트입니다.

`ask(instructions, source)`는 이 노트북에서 만드는 함수입니다. 첫 인자는 SDK의 `instructions`로, 둘째 인자는 `input`으로 전달합니다. 이제 독자만 바꾸고 싶다면 첫 인자를 고치면 됩니다. 두 인자가 모두 문자열이라 순서를 뒤집어도 호출 자체는 성공할 수 있습니다. 답이 엉뚱할 때는 어떤 글이 어느 칸에 들어갔는지 먼저 확인하세요.

### 앞 장에서 마지막으로 보낸 공지 다시 열기

8장에서 저장한 공지를 읽습니다. 회원용과 진행자용 두 응답을 비교할 때는 아래 notice의 값을 고정하세요. 예제 기록과 내 파일이 다르면 날짜와 장소는 내 원문을 기준으로 봅니다.

In [ ]:
example_notice = "5월 21일 금요일 오후 4시에 가온관 305호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요."
notice = read_work("notice-08.json", {"source": example_notice})["source"]
print(notice)

5월 21일 금요일 오후 4시에 가온관 305호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.


실습 파일이 없으면 305호가 적힌 예시가 나옵니다. 다음 ask 호출에서 두 번째 인자로 이 글이 들어갑니다.

In [ ]:
def ask(instructions, source):
    # 요청마다 새 원문을 보내므로 앞 호출의 장소를 가져오지 않습니다.
    reply = client.responses.create(
        model=model, instructions=instructions, input=source, store=False
    )
    return reply.output_text  # 실습에서 비교할 답변 문장만 꺼냅니다.

In [ ]:
print(ask("이 공지를 정리해 주세요.", notice))

출력에서 일시·장소와 각 사람이 해야 할 일을 찾아보세요. 입력을 그대로 썼다면 장소는 305호입니다. 짧게 요약하더라도 대상 독자를 정하지 않으면 신입 회원에게 진행자의 준비 업무까지 전달할 수 있습니다.

In [ ]:
member_prompt = '소리채집 동아리의 신입 회원에게 지금 해야 할 행동을 한 문장으로 알려 주세요. 공지에 있는 사실만 사용하세요.'
print(ask(member_prompt, notice))

회원에게 필요한 행동인 이어폰 준비가 남았나요? “신입 회원에게”는 독자를 정하고, “지금 해야 할 행동”은 고를 내용을 정하며, “한 문장”은 답의 길이를 정합니다. 자신이 바꾼 공지를 보냈다면 그 원문에 있는 준비물로 확인하세요.

### 실습 · 진행자에게 보내는 한 문장

같은 공지로 진행자의 문자 안내를 만드세요. 신입 회원용 지시를 복사해 독자만 고칩니다. 직접 호출한 뒤 앞 답에 없던 준비 행동을 짚으세요. 이어폰 준비와 기기 배치가 섞였을 때는 어떤 지시를 더하면 좋을지 적어 보세요.

In [ ]:
organizer_prompt = ""  # 진행자가 해야 할 일을 한 문장으로 요청하는 지시를 쓰세요.
# ask(organizer_prompt, notice)를 호출하세요. 같은 공지에서 회원용 답과 다른 행동이 나오는지 확인하세요.

<details><summary>모범 프롬프트와 응답 살펴보기</summary><pre><code>organizer_prompt = &#x27;소리채집 동아리의 진행자에게 지금 해야 할 행동을 한 문장으로 알려 주세요. 공지에 있는 사실만 사용하세요.&#x27;
print(ask(organizer_prompt, notice))</code></pre><p>이 지시로 받은 실제 응답</p><pre>모임 20분 전인 오후 3시 40분에 가온관 201호에 도착하여 녹음기를 책상마다 한 대씩 놓으세요.</pre><p>3시 40분은 오후 4시에서 20분을 뺀 시각입니다. 책상마다 한 대씩 배치하라는 조건도 보존했습니다. 응답이 다른 역할의 일까지 담으면 “진행자가 직접 맡은 행동만 포함하세요”를 더해 다시 시험해 봅니다.</p></details>

### 내 프롬프트를 고칠 때 무엇을 남길까요?

같은 원문과 모델에서 한 번에 조건 하나를 바꾸면 어느 지시를 시험했는지 설명할 수 있습니다. 단 한 번의 답으로 효과를 단정하기는 어렵습니다. 다른 공지에서도 같은 문제가 되풀이되는지 확인하세요.

| 받은 답에서 발견한 문제 | 다음 요청에 넣어 볼 말 | 다시 읽을 부분 |
|---|---|---|
| 일정까지 빠질 만큼 짧아짐 | “일시·장소를 남긴 한 문장으로 써 주세요.” | 날짜와 호수가 함께 남았나요? |
| 너무 딱딱한 안내문 | “신입 회원 단체 채팅방에 보낼 존댓말로 써 주세요.” | 바로 보낼 수 있는 말투인가요? |
| 누가 할 일인지 섞임 | “진행자가 직접 준비할 일만 시간 순서로 써 주세요.” | 이어폰 준비가 진행자의 할 일로 바뀌었나요? |

짝이 고른 문제 하나만 고쳐 보세요. 활동지에 최초 지시, 바꾼 구절, 실제 답에서 달라진 문장을 남깁니다.

### 방 이름을 모를 때는 무엇을 더 보내야 할까요?

M1에는 ‘집결지는 지난번과 같다’고만 적혀 있습니다. 지난 산책 안내 M0를 담당자만 갖고 있다면 모델도 이 표현에서 방 이름을 확인할 수 없습니다. 먼저 M1만 넣어 “집결 위치와 근거 구절을 알려 줘. 건물명이 없으면 확인 질문 한 개를 작성해 줘”라고 요청합니다. 그다음에는 **같은 지시**에 두 문서를 함께 넣습니다. ‘지난 안내’와 ‘이번 메모’라는 제목을 붙여 시점이 섞이지 않게 하세요.

이때 나온 장소와 근거를 나란히 놓아 보세요. M0에는 옛 장소가 있고, M1에는 이번에도 같은 장소라는 연결 문장이 있습니다. 둘 중 하나만 주면 이번 장소를 확정하기 어렵습니다. 다른 정보를 더 많이 주는 것보다 **지금 판단에 필요한 두 문장**을 골라 주는 것이 중요합니다.

In [ ]:
# 두 글의 시점을 제목으로 구분해 한 요청에 함께 보냅니다.
past_and_current = """[지난 산책 안내] 지난 녹음 산책 때는 솔빛관 1층 안내대 앞에서 모였습니다.
[이번 회의 메모] 6월 14일 오전 10시에 녹음 산책을 시작한다. 집결지는 지난번과 같다."""
place_request = "이번 집결지를 한 줄로 쓰고, 판단에 사용한 원문 구절을 두 글에서 각각 하나씩 옮겨 주세요."
print(ask(place_request, past_and_current))

먼저 장소를 보고, 그다음 ‘솔빛관 1층 안내대’와 ‘지난번과 같다’가 모두 근거에 있는지 확인하세요. 답에 정문 오른쪽이라고 나오면 이번 셀의 짧은 원문에는 그 방향이 있는지 다시 읽습니다. 전체 M0에 있는 사실도 이번 호출에 넣은 발췌문에는 없을 수 있습니다.

이 셀에는 저장한 실행 응답이 없습니다. 자신의 계정으로 받은 문장과 원문에서 확인한 근거를 활동지에 남기세요.

### 길이와 사실이 충돌할 때

M1로 “문자 한 줄, 25자 이내”를 요청했다고 합시다. 신청 마감과 신청 면제 조건을 남기지 못하면 이미 신청한 사람이 두 번 신청할 수 있습니다. 문자 수는 전달 형식이고 마감과 면제 조건은 회원의 행동을 바꾸는 사실입니다. **길어져도 보존해야 하는 내용**을 지정해야 모델이 무엇을 줄일지 판단할 단서를 얻습니다.

실험은 같은 M1로 합니다. 첫 지시는 “회원들에게 신청을 25자 이내로 안내해 줘”입니다. 실제 답에서 마감·예외·신청 위치를 찾으세요. 두 번째 지시에서는 빠진 항목 이름을 적고 두 문장까지 허용합니다. “간단히, 자세히, 완벽하게” 같은 말만 더한 요청과 어느 문장에서 차이가 나는지 짝과 읽습니다. 두 번째 답이 더 길어도 불필요한 설명이 늘어난 것인지, 필요한 정보가 돌아온 것인지는 직접 판정해야 합니다.

### 초안의 날짜와 행동을 원문에 맞게 고치기

**수정할 초안(E1)**

> “6월 12일 오후 6시에 솔빛관에서 녹음 산책이 시작됩니다. 기존 신청자를 포함하여 전원 다시 신청해 주세요. 비가 오면 자동 취소됩니다.”

**회의 메모에서 발췌**

> 6월 14일 오전 10시에 녹음 산책을 시작한다. 참가 신청은 6월 12일 오후 6시까지 동아리 게시판에서 받는다. 지난 활동에서 이미 신청서를 낸 회원은 다시 신청할 필요가 없다. 집결지는 지난번과 같다.

> 비가 오면 출발 여부를 당일 오전 8시 게시판에 공지한다.

초안과 발췌문을 함께 보내세요. “신청 마감과 산책 시작을 구분하고, 재신청 조건과 비가 올 때 할 일을 바로잡아 줘. 메모에서 확인할 수 있는 장소만 써 줘.” 답에서 ‘솔빛관’을 이번 메모만으로 확인할 수 있는지도 살펴보세요.

<details><summary>편집자가 남긴 수정 지시와 판정 펼치기</summary><p>수정 지시 예시입니다. “아래 회의 메모를 근거로 초안을 고쳐 줘. 날짜마다 무엇의 마감인지 무엇의 시작인지 밝혀 줘. 다시 신청할 필요가 없는 회원을 남기고, 비가 올 때 이미 정해진 행동만 써 줘.”</p><p>회원 안내 예시입니다. “기존 신청자를 제외한 회원은 6월 12일 오후 6시까지 게시판에 신청하세요. 산책은 6월 14일 오전 10시에 시작하며 비가 오면 당일 오전 8시 게시판에서 출발 여부를 확인하세요.” 사람이 작성한 예시로, 문장 수를 더 줄여도 세 조건이 그대로면 근거 있는 답입니다. 비가 오면 게시판에서 출발 여부를 확인합니다. 메모에는 ‘지난번과 같다’고만 적혀 있어 건물 이름은 알 수 없습니다.</p></details>

### 마감 상태만 바꿔 지시문 점검하기

다음 세 공지에 같은 지시를 보내세요. “회원에게 지금 할 일을 한 문장으로 알려 줘.”

- “참가 신청은 오늘 오후 6시까지 게시판에서 받습니다.”
- “참가 신청을 마감했습니다.”
- “참가 신청 방법과 기한은 추후 공지합니다.”

두 번째 답에서도 신청을 권하거나 세 번째 답에 없는 마감 시각이 생겼나요? “신청 가능 여부와 기한은 공지에서 확인할 수 있는 만큼만 써 줘”를 덧붙이고 다시 비교하세요. 짝이 새로 쓴 공지에서도 확인해 보세요.

In [ ]:
# 같은 메모를 두고 지시만 바꿔 봅니다. 첫 지시를 쓰고 마지막 줄의 주석을 푸세요.
memo = """6월 14일 오전 10시에 녹음 산책을 시작한다. 참가 신청은 6월 12일 오후 6시까지 동아리 게시판에서 받는다. 지난 활동에서 이미 신청서를 낸 회원은 다시 신청할 필요가 없다. 집결지는 지난번과 같다. 진행자는 시작 30분 전에 도착해 이름표를 꺼내 둔다. 비가 오면 출발 여부를 당일 오전 8시 게시판에 공지한다."""
first_instruction = ""
# print(ask(first_instruction, memo))

<details><summary>초안과 수정 요청의 코드 예시 보기</summary><pre><code># 처음 쓴 지시는 회원이 무엇을 판단해야 하는지 충분히 밝히지 않았습니다.
first_instruction = &quot;회원에게 보낼 짧은 문자를 써 주세요.&quot;
print(ask(first_instruction, memo))

# 판단 기준을 한 줄 더해 같은 원문으로 비교합니다.
revised_instruction = first_instruction + &quot; 신청 가능 여부를 원문으로 판정하고 재신청 예외와 마감을 남겨 주세요.&quot;
print(ask(revised_instruction, memo))</code></pre><p>자신의 답변을 활동지에 붙여 두세요. M1에서 기존 신청자는 다시 신청할 필요가 없다는 점이 두 번째 답에 살아 있는지 읽고, 짝이 쓴 ‘신청이 끝난 공지’에도 같은 지시를 보내 보세요. 위 코드는 실행하면 두 번 호출합니다.</p></details>

### 회원용 지시와 기준 원문을 한 쌍으로 저장하기

같은 공지에서 진행자의 할 일을 뽑아냈다면 organizer_prompt를 보관하세요. 그 과제를 건너뛰면 앞에서 실행한 회원용 지시를 사용합니다.

In [ ]:
chosen_rule = organizer_prompt.strip() or member_prompt
save_work("notice-09.json", {"source": notice, "instruction": chosen_rule})
print(chosen_rule)

소리채집 동아리의 신입 회원에게 지금 해야 할 행동을 한 문장으로 알려 주세요. 공지에 있는 사실만 사용하세요.


출력된 문장이 내가 맡긴 일입니다. 10장에서는 이 문장을 한 번 읽고, 같은 원문에 날짜·장소를 뽑는 새 지시를 적용합니다.

## 10장 · 예시로 출력 형식을 보여 주고 빈 정보를 처리하기

지시문에 입력과 출력의 짝을 써서 원하는 형식을 보여 줄 수 있습니다. 예를 들어 “입력: 사과 두 개 / 출력: 2”를 먼저 제시하고 “배 세 개”를 보내면 기대하는 답은 숫자 3입니다. 값이 없을 때 어떤 표기를 쓸지도 함께 정합니다.

새 커널에서는 맨 위 파일 준비 셀, 8장의 OpenAI() 클라이언트 준비 셀, 9장의 def ask(...) 셀을 먼저 실행하세요. 예제 요청은 건너뛰어도 됩니다. 9장 파일을 받지 못했다면 아래 읽기 셀의 짧은 공지로 같은 함수를 시험할 수 있습니다.

### 예시와 실제 입력을 분리해 읽기

형식을 보여 주는 예시는 지시문 안에 두고, 이번에 읽을 텍스트는 별도 인자로 보냅니다. 모델을 부르기 전에 두 값이 어떻게 구분되는지 확인하세요.

In [ ]:
example_rule = "수량을 숫자로만 써. 예시: 사과 두 개 → 2. 수량이 없으면 미상."
new_text = "배 세 개"
print(example_rule)
print(new_text)

수량을 숫자로만 써. 예시: 사과 두 개 → 2. 수량이 없으면 미상.
배 세 개


여기서 출력한 2는 예시 속 사과의 수량입니다. 실제 요청에는 배를 세 개 주었으므로 기대하는 응답은 3입니다. 다음 예제에서는 장소가 빠진 새 공지로 같은 구분을 적용합니다.

### 같은 원문에서 출력할 항목 바꾸기

9장의 원문과 지시를 열고 옆에 적어 두세요. 앞 장에서는 그 지시로 회원이나 진행자가 할 일을 골랐습니다. 이번에는 같은 공지에서 장소가 빠지면 어떤 표기를 쓸지 정합니다.

In [ ]:
example_case = {"source": "5월 21일 금요일 오후 4시에 가온관 305호에서 모입니다.", "instruction": "회원이 할 일을 한 문장으로 안내하세요."}
previous_case = read_work("notice-09.json", example_case)
print(previous_case["instruction"])
print(previous_case["source"])

소리채집 동아리의 신입 회원에게 지금 해야 할 행동을 한 문장으로 알려 주세요. 공지에 있는 사실만 사용하세요.
5월 21일 금요일 오후 4시에 가온관 305호에서 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.


첫 줄은 이전 작업이고 둘째 줄은 새 작업에도 쓰는 글입니다. 짧은 예제에서 빈 항목을 처리하는 법을 먼저 익힌 뒤 이 원문에 적용하세요.

### 출력 예시와 누락값 규칙을 지시에 넣기

지시문에 원하는 항목, 쓰는 순서, 값이 없을 때 표시할 말을 정합니다. 예시의 장소는 형식을 보여 주는 값입니다. 새 원문에 장소가 없을 때는 예시의 방 번호를 복사하지 않도록 `미정`이라고 쓰게 합니다. 다음 셀에서 예시가 있는 지시문과 실제로 읽힐 `no_place`를 구별해 보세요.

In [ ]:
no_place = '5월 21일 금요일 오후 4시에 소리채집 동아리 첫 모임을 엽니다. 신입 회원은 이어폰을 챙겨 오세요. 진행자는 모임 20분 전에 도착해 녹음기를 책상마다 한 대씩 놓아 주세요.'
extract_prompt = """공지에서 모임 일시와 모임 장소를 뽑아 두 줄로 작성하세요. 원문에 없는 항목은 ‘미정’으로 적으세요. 제출 마감과 모임 일시를 구분하세요.
입력 예시: 6월 2일 오후 2시, 별관 101호에서 모입니다.
출력 예시:
모임 일시: 6월 2일 오후 2시
모임 장소: 별관 101호"""
print(ask(extract_prompt, no_place))

모임 일시: 5월 21일 금요일 오후 4시  
모임 장소: 미정


출력 예시에는 별관 101호가 있지만 실제 답의 장소는 ‘미정’입니다. 여기서 ‘미정’은 공지에 장소가 적혀 있지 않다는 표기입니다. 주최자가 아직 장소를 정하지 않았는지는 알 수 없습니다. 예시는 답의 모양을 알려 줍니다. 지금 처리하는 문서의 사실은 input에서 가져와야 합니다. 예시에 쓴 날짜나 장소가 실제 응답에 튀어나온다면 예시와 원문을 더 분명하게 구분하고 장소 없는 예시를 하나 더 제공해 보세요.

### 실습 · 제출 마감도 모임 일시일까요?

아래 공지에는 5월 28일 오전 11시라는 시각이 등장합니다. 무엇을 하기 위한 시각인지 먼저 읽고, 예시가 있는 지시를 새 공지에 보내세요. 모델이 이 값을 모임 일시로 옮겼다면 “무엇의 일시인지”를 직접 설명한 지시로 바꿔 재실행합니다. 짝에게는 다른 입력 예시 하나를 직접 써서 건네세요.

In [ ]:
submission_notice = '5월 28일 금요일 오전 11시까지 지난 모임에서 녹음한 소리를 동아리 게시판에 올려 주세요. 파일 제목에 녹음 장소를 넣어 주세요. 다음 주 모임에서 함께 들을 파일은 진행자가 두 개 고릅니다.'
# extract_prompt와 submission_notice로 답을 받아 두 칸을 원문과 대조하세요.
# 오전 11시는 무엇의 마감인가요? 모임 시간 칸에 11시가 들어갔다면 원문에서 그 근거를 찾아보세요.

<details><summary>모범 프롬프트와 응답 살펴보기</summary><pre><code>print(ask(extract_prompt, submission_notice))</code></pre><p>이 지시로 받은 실제 응답</p><pre>모임 일시: 미정  
모임 장소: 미정</pre><p>두 칸이 모두 미정입니다. 오전 11시는 제출 마감이고 다음 모임의 일시와 장소는 적혀 있지 않습니다. 제출 기한을 안내하는 과제라면 먼저 요청하는 항목을 ‘제출 마감’으로 바꿔야 합니다.</p></details>

### 모임 시작과 신청 마감을 다른 칸에 담기

**운영진 회의 메모(M1)에서 발췌**

> 6월 14일 오전 10시에 녹음 산책을 시작한다. 참가 신청은 6월 12일 오후 6시까지 동아리 게시판에서 받는다. […] 집결지는 지난번과 같다.

메모에서 ‘모임 시작’과 ‘참가 신청 마감’ 두 칸에 들어갈 시각을 찾아보세요. ‘집결지’에는 무엇을 쓸 수 있을까요? 메모에는 지난번과 같다고만 적혀 있으므로 건물 이름을 쓰려면 지난 안내가 필요합니다.

“장소는 추후 결정합니다”라고 적힌 공지와 장소를 아예 언급하지 않은 공지도 비교하세요. 앞의 경우에는 아직 장소를 정하지 않았다는 사실까지 알 수 있습니다.

### 학생이 만든 예시가 답을 오염시키는지 확인하기

직접 예시를 만들어 보세요. ‘접수 마감만 있는 공지’를 예시 입력으로 쓰고, 출력에서 모임 시작은 ‘미기재’로 둡니다. 예시에는 일부러 새 단어와 새 장소를 사용하세요. 모델이 그 장소를 다른 공지에도 옮겼다면 어디서 나온 값인지 추적하기 쉽습니다. 지시는 그대로 두고 **예시를 뺀 요청과 넣은 요청**을 각각 보냅니다. 두 응답에서 어느 쪽이 새 공지의 사실을 더 잘 보존했는지 확인하세요.

**비교할 공지(E2)**

> 7월 활동은 신청을 마감했습니다. 취소 신청은 7월 3일 정오까지 게시판에서 받습니다. 참가자 명단은 7월 4일에 게시합니다. 활동 당일에 사용할 집결지는 확정되는 대로 알려 드립니다.

같은 지시에 이 공지를 넣으세요. 7월 3일은 취소 신청 마감이고 7월 4일은 명단 게시일입니다. 두 날짜가 참가 신청 마감 칸에 들어갔다면, 어떤 신청의 마감인지 지시에 밝혀 다시 시험하세요.

<details><summary>두 공지의 접수표 대조하기</summary><p>M1의 모임 시작은 6월 14일 오전 10시, 참가 신청 마감은 6월 12일 오후 6시입니다. M1만 받았다면 집결지는 “지난번과 같음, 구체 위치 확인 필요”가 적절합니다. M0까지 받으면 솔빛관 1층 안내대 앞이라고 쓸 수 있습니다.</p><p>E2는 7월 활동 신청이 이미 끝났다고만 밝힙니다. 참가 신청 마감 시각은 미기재입니다. 7월 3일 정오는 취소 신청 마감이고 7월 4일은 명단 게시일입니다. ‘집결지는 추후 안내’라는 상태는 유지하고 구체적인 장소명은 비워 둡니다.</p></details>

In [ ]:
# 자신이 만든 입력·출력 예시를 아래의 빈칸에 넣고 E2에서 세 칸을 확인합니다.
schema_instruction = """모임 시작, 참가 신청 마감, 집결지를 세 줄로 써 주세요.
원문에서 확인할 수 없는 값의 표기:
내 입력 예시:
그 예시에 해당하는 세 줄:"""
comparison_notice = "7월 활동은 신청을 마감했습니다. 취소 신청은 7월 3일 정오까지 게시판에서 받습니다. 참가자 명단은 7월 4일에 게시합니다. 활동 당일에 사용할 집결지는 확정되는 대로 알려 드립니다."
# print(ask(schema_instruction, comparison_notice))

<details><summary>입출력 예시를 넣은 지시 한 가지</summary><pre><code>schema_instruction = &quot;&quot;&quot;모임 시작 / 참가 신청 마감 / 집결지를 세 줄로 써 줘.
각 값은 새 원문을 근거로 써 줘. 적힌 값이 없으면 &#x27;미기재&#x27;라고 해 줘.
예시 입력: 이번 모임은 9월 2일 오전 11시에 시작한다. 9월 1일은 준비 회의 날짜다.
예시 답:
모임 시작: 9월 2일 오전 11시
참가 신청 마감: 미기재
집결지: 미기재&quot;&quot;&quot;</code></pre><p>예시에서 9월 1일을 어느 칸에도 넣지 않은 이유를 설명할 수 있어야 합니다. E2를 보냈을 때 9월 2일이나 9월 1일이 나타났다면 예시와 새 입력을 혼동한 것입니다. 모델에 보낸 실제 지시와 원문을 확인하세요.</p></details>

### 개인 실습 · 반납 장소를 한 문장으로 안내하기

**반납함 점검 공지(S1)**

> 7월 8일 오전 9시부터 정오까지 솔빛도서관 동문 무인 반납함을 점검합니다. 그동안 책을 반납할 이용자는 1층 안내 데스크를 이용해 주세요. 도서관 열람실은 정상 운영합니다. 예약 도서 수령은 이번 점검의 영향을 받지 않습니다.

7월 8일 오전 10시에 책을 반납할 이용자에게 문자를 보내세요. `ask(내_지시, 시설_원문)`으로 안내를 작성한 뒤 짝에게 답만 보여 주세요. 어느 반납함을 쓸 수 없는지, 대신 어디로 가야 하는지 물어보세요. 열람실까지 문을 닫는다고 읽힌다면 점검 대상을 정확히 밝혀 다시 요청하세요.

<details><summary>이용자에게 보낼 문자 예시</summary><p>“7월 8일 오전 9시부터 정오까지 동문 무인 반납함을 점검하니 1층 안내 데스크에서 책을 반납해 주세요.”</p><p>점검 대상은 동문 무인 반납함입니다. 열람실과 예약 도서 수령은 평소처럼 이용할 수 있습니다.</p></details>

### responses.parse로 항목별 값 받기

사람에게 문자로 보낼 때는 위처럼 두 줄의 답을 읽으면 충분합니다. 일정 화면의 날짜 칸과 장소 칸에 각각 값을 넣으려면 두 항목을 정해 둔 구조로 받을 수 있습니다. 아래 Meeting은 답의 칸을 선언합니다. str은 문자열이고, None을 함께 허용한 칸에는 값이 없을 수 있습니다. Pydantic은 이 칸들의 자료형을 확인합니다.

`responses.parse(..., text_format=Meeting)`은 답을 코드에서 필드별로 읽을 때 씁니다. `Meeting`에는 `when`과 `where` 두 칸이 있고, 각 값은 문자열 또는 `None`이 될 수 있습니다. 응답의 `output_parsed`가 있으면 점 뒤에 필드 이름을 써서 읽습니다. “장소 미정”이라는 문자열과 실제 값이 비어 있는 `None`을 구별해 보세요.

### 문자열과 None을 받는 필드 선언하기

먼저 모델 호출 없이 Pydantic이 어느 값을 필드에 보관하는지 확인합니다. `str | None`은 문자열과 값 없음 중 하나를 허용합니다. 아래 선언에는 기본값이 없으므로 두 필드 이름을 모두 전달합니다.

In [ ]:
from pydantic import BaseModel

class MeetingFields(BaseModel):
    when: str | None
    where: str | None

meeting = MeetingFields(when="오전 10시", where=None)
print(meeting.when, meeting.where)

오전 10시 None


두 번째 값 `None`은 장소를 모른다는 뜻입니다. 다음 셀에서는 같은 구조를 SDK의 `text_format`에 건네고, 모델이 작성한 값을 `output_parsed`에서 읽습니다.

In [ ]:
from pydantic import BaseModel

class Meeting(BaseModel):
    # 공지에서 찾지 못한 항목은 값이 없어도 됩니다.
    when: str | None
    where: str | None

parsed = client.responses.parse(
    model=model,
    instructions="모임 일시를 when에, 장소를 where에 넣으세요. 공지에 없으면 null로 두세요.",
    input=no_place, text_format=Meeting, store=False,
)
if parsed.output_parsed is not None:
    print(parsed.output_parsed.when)
    print(parsed.output_parsed.where)  # 장소가 없으면 Python에서 None입니다.
else:
    print("구조화된 답을 받지 못했습니다.")

5월 21일 금요일 오후 4시
None


### 실습 · 내 공지를 두 줄로 바꿔 다음 대화에 넘기기

앞의 예에서 만든 extract_prompt와 9장 원문을 send 함수에 넘길 수 있도록 보관합니다. 먼저 ask를 써서 두 줄을 받아 보세요. 장소를 한 번 지운 원문과 비교하고, 지운 뒤에도 305호가 나온다면 그 지시에 누락값 규칙이 있는지 확인하세요.

In [ ]:
reviewed_notice = previous_case["source"]
# 아래에서 지시와 원문을 ask에 넘겨 두 줄을 읽으세요.
# print(ask(...))

<details><summary>호출 코드와 출력 확인 방법</summary><pre><code>print(ask(extract_prompt, reviewed_notice))</code></pre><p>기본 원문에 대한 기대 형식은 첫 줄 ‘모임 일시: 5월 21일 금요일 오후 4시’, 둘째 줄 ‘모임 장소: 가온관 305호’입니다. 모델이 쓴 조사와 공백은 달라질 수 있습니다. 장소 자체가 빠진 원문을 보냈다면 장소는 미정이어야 합니다.</p></details>

In [ ]:
save_work("notice-10.json", {"source": reviewed_notice, "instruction": extract_prompt})
print(read_work("notice-10.json", {})["source"] == reviewed_notice)

True


True면 원문과 지시가 파일에 남았습니다. 11장에서는 이 원문을 첫 메시지로 보내고 12장의 화면에서 답을 다시 엽니다.

자료형에 맞는 장소 문자열이라도 공지에 없는 101호라면 잘못된 답입니다. 처음에는 날짜와 장소를 원문에서 직접 대조하세요. 다음 수업에서는 상대가 앞에서 말한 날짜를 정정하는 대화를 다룹니다. 지금까지 ask를 부를 때마다 모델이 받은 공지는 그 호출의 input 하나뿐이었습니다.

2026년 11월 26일 Python 3.12, openai 3.35.1로 기존 공지 C1~C5 예시와 접힌 풀이의 대표 요청 8건을 실행했습니다. SDK 참고 문서 · [텍스트 생성](https://developers.openai.com/api/docs/guides/text) · [구조화된 출력](https://developers.openai.com/api/docs/guides/structured-outputs)

### 개념 참고 자료

[Google의 언어 모델 기초 설명](https://developers.google.com/machine-learning/crash-course/llm/transformers) · [트랜스포머를 제안한 논문, Attention Is All You Need](https://arxiv.org/abs/1706.03762)